In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Italy Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'IT'
NUTS2 = 'Veneto'

In [4]:
YEAR = 2023
MONTH = 'May'
PERIOD = '2nd'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,population,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,11.79324,45.35865,2023-05-16,veneto,abano terme,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,"19,349",46.86670,0.384022,0.207582,-0.305785,-0.207582,0.374553,0.192710,-0.298535,-0.192710,0.132372,0.092062,0.111451,0.092062,16.180,22.71,9.650,7.626000,0.600000,8.080000,-0.791538,18.365714,6.326364,21.500769,6.950000,153.235893,196.265873,671.012194,8982.126981,2074.539971,3,170.409233,10.722186,179.966181,0.0,1.063084,31,98.0,36,98.0,30,98.0,12,12,1,6,7,2,0,222,0,0
1,12.04199,45.06525,2023-05-16,veneto,adria,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,"20,233",46.64640,0.301307,0.085303,-0.282241,-0.085303,0.356996,0.173485,-0.313694,-0.173485,0.068094,0.063588,0.057293,0.063588,15.572,26.19,4.954,6.377211,0.808571,8.476395,-1.590923,16.728553,4.271543,21.836459,5.637644,206.454879,221.728639,1038.975884,6701.380530,1644.125334,2,145.603087,-1.354496,180.070371,0.0,1.206604,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,190,0,0
2,10.77673,45.55680,2023-05-16,veneto,affi,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,"2,297",46.81410,0.215253,0.343317,-0.097417,-0.343317,0.286031,0.303942,-0.179514,-0.303942,0.070778,0.039374,0.082097,0.039374,16.180,21.15,11.210,5.821667,0.738000,8.631538,0.720667,17.207647,3.983750,18.918421,6.878889,53.175942,107.485377,296.699086,5306.025387,738.794180,6,207.641691,201.821006,178.733735,0.0,1.815834,31,84.0,30,84.0,30,84.0,10,10,1,6,6,2,0,72,0,0
3,11.96539,45.17531,2023-05-16,veneto,agna,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,"3,400",46.73306,0.291590,-0.007849,-0.311519,0.007849,0.372874,0.089191,-0.373604,-0.089191,0.108124,0.099352,0.078616,0.099352,17.710,23.55,11.870,4.870000,0.610000,7.752222,-2.034615,15.791176,4.124000,21.578571,5.119231,175.660153,212.282110,950.761858,16631.850238,1165.197945,3,121.193871,0.122498,180.188821,0.0,3.769479,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,124,0,0
4,12.04755,46.30297,2023-05-16,veneto,agordo,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,"4,249",47.84463,0.317522,-0.165724,-0.385563,0.165724,0.384072,-0.044801,-0.399386,0.044801,0.045556,0.057840,0.025550,0.057840,9.310,20.95,-2.330,1.921429,-4.077692,5.623333,-2.298000,9.285385,-0.998333,11.544545,0.444545,158.277616,193.132945,742.683124,15023.566702,684.464446,18,124.001040,1470.386475,152.701532,0.0,2.289867,15,91.0,10,97.0,10,97.0,5,5,7,1,1,2,0,23,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)


In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,11.73897,44.96915,polesella,16,5,2023,1.108711e-02
1,11.79794,45.06527,rovigo,16,5,2023,1.375452e-03
2,12.64550,45.53937,jesolo,16,5,2023,1.373833e-03
3,12.54296,45.67127,noventa da piave,16,5,2023,1.270038e-03
4,12.15513,45.49083,spinea,16,5,2023,1.170599e-03
...,...,...,...,...,...,...,...
566,11.39118,45.89987,rotzo,16,5,2023,8.338114e-11
567,12.42616,46.50321,lozzo da cadore,16,5,2023,7.434996e-11
568,12.05493,46.45349,selva da cadore,16,5,2023,5.985187e-11
569,11.71465,46.07125,lamon,16,5,2023,4.990256e-11


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.2916874171585726, 0.7128345091584661, 0.8345479844283598, 0.9597101609376664, 0.983219513129097, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,11.73897,44.96915,polesella,16,5,2023,1.108711e-02,0
1,11.79794,45.06527,rovigo,16,5,2023,1.375452e-03,0
2,12.64550,45.53937,jesolo,16,5,2023,1.373833e-03,0
3,12.54296,45.67127,noventa da piave,16,5,2023,1.270038e-03,0
4,12.15513,45.49083,spinea,16,5,2023,1.170599e-03,0
...,...,...,...,...,...,...,...,...
566,11.39118,45.89987,rotzo,16,5,2023,8.338114e-11,0
567,12.42616,46.50321,lozzo da cadore,16,5,2023,7.434996e-11,0
568,12.05493,46.45349,selva da cadore,16,5,2023,5.985187e-11,0
569,11.71465,46.07125,lamon,16,5,2023,4.990256e-11,0


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results